In [1]:
import subprocess

ALPHABET : list[str] = ['A', 'C', 'G', 'T']
WORD_LENGTH : int = 8

In [2]:
def Check_CG_Count(word : str) -> bool:
    if(word.count('C') + word.count('G') == 4):
        return True
    return False

In [3]:
def Check_4_Position_Difference(word_x : str, word_y : str) -> bool:
    diffCount = 0
    for i in range(WORD_LENGTH):
        if(word_x[i] != word_y[i]):
            diffCount += 1

    if(diffCount >= 4):
        return True
    return False

In [4]:
def Check_RC_Difference(word_x : str, word_y :str) -> bool:
    R_word = Get_Reverse_Word(word_x)
    C_word = Get_WCC_Word(word_y)

    if(Check_4_Position_Difference(R_word, C_word)):
        return True
    return False

def Get_Reverse_Word(word : str) -> str:
    return word[::-1]

def Get_WCC_Word(word: str) -> str:
    complements = {
        'A': 'T',
        'T': 'A',
        'C': 'G',
        'G': 'C',
    }

    return ''.join(complements[letter] for letter in word)

In [5]:
def GenerateAllEligibleWords() -> list[str]:
    words = [""]
    for i in range(WORD_LENGTH):
        words = GenerateAllEligibleWordsHelper(words)

    return [word for word in words if(Check_CG_Count(word) and Check_RC_Difference(word, word))]

def GenerateAllEligibleWordsHelper(array : list[str]) -> list[str]:
    out = []
    for subword in array:
        for i in ALPHABET:
            out.append(subword + i)
    return out

def GenerateVariablesForWords(words : list[str]) -> dict[str , int]:
    return {
        word: index
        for index, word in enumerate(words, start = 1)
        }

In [ ]:
def GetRuleClauses(words : list[str], wordVariables : dict[str , int]) -> list[tuple[int, int]]:

    clauses = []
    for word_x in words:
        for word_y in words:
            if(word_x == word_y):
                continue
            if(not (Check_4_Position_Difference(word_x, word_y) and Check_RC_Difference(word_x, word_y))):
                clauses.append((-wordVariables[word_x], -wordVariables[word_y]))

    return clauses

In [ ]:
def GetSizeClauses(wordVariables : dict[str, int], targetSize : int) -> tuple[list[tuple[int, ...]], int]:
    clauses = []
    selectedVariables = list(wordVariables.values())
    wordCount = len(selectedVariables)
    nextVariable = max(selectedVariables, default = 0) + 1

    trueLiteral = nextVariable
    nextVariable += 1
    clauses.append((trueLiteral,))

    falseLiteral = -trueLiteral

    dp = [[falseLiteral] * (targetSize + 2) for _ in range(wordCount + 1)]

    for i in range(wordCount + 1):
        dp[i][0] = trueLiteral

    for i in range(1, wordCount + 1):
        selectedVariable = selectedVariables[i - 1]

        for j in range(1, targetSize + 2):
            currentState = nextVariable
            nextVariable += 1
            dp[i][j] = currentState

            alreadyReached = dp[i - 1][j]
            previousLevel = dp[i - 1][j - 1]

            clauses.append((-alreadyReached, currentState))
            clauses.append((-previousLevel, -selectedVariable, currentState))
            clauses.append((-currentState, alreadyReached, previousLevel))
            clauses.append((-currentState, alreadyReached, selectedVariable))

    clauses.append((dp[wordCount][targetSize],))
    clauses.append((-dp[wordCount][targetSize + 1],))

    return clauses, nextVariable - 1


In [ ]:
def Glucose(variableCount : int, clauses : list[tuple[int, ...]], solverPath : str) -> subprocess.CompletedProcess[str]:
    inputPath = "glucoseClauses.txt"
    with open(inputPath, 'w') as file:
        file.write(f"p cnf {variableCount} {len(clauses)}\n")
        for clause in clauses:
            file.write(f"{' '.join(map(str, clause))} 0\n")

    return subprocess.run(
        [solverPath, '-model', '-verb=0', inputPath],
        capture_output=True,
        text=True,
    )


In [ ]:
def ParseGlucoseModel(result: subprocess.CompletedProcess[str]) -> set[int] | None:
    if result.returncode == 20:
        return None

    selectedVariables = set()

    for line in result.stdout.splitlines():
        tokens = line.split()

        if(not tokens or tokens[0] != 'v'):
            continue

        for token in tokens[1:]:
            literal = int(token)

            if(literal == 0):
                break

            if(literal > 0):
                selectedVariables.add(literal)

    return selectedVariables

In [ ]:
def DecodeGlucoseOutput(parsedModel : set[int] | None, setSize : int) -> None:
    if(parsedModel is None):
        return print("Glucose didnt find any solutions.")

    selectedWords = [word for word, variable in wordVariables.items() if variable in parsedModel]
    finalSize = len(selectedWords)

    if(finalSize != setSize):
        return print(f"Couldnt find a model with target size.")
    
    return print(f"Found solution for words:\n{'\n'.join(selectedWords)}")
    

In [ ]:
def Run(target_set_size : int, solverPath : str, variableCount : int, clauses : list[tuple[int, ...]]) -> None:
    glucoseOutput = Glucose(variableCount, clauses, solverPath)
    parsed = ParseGlucoseModel(glucoseOutput)
    DecodeGlucoseOutput(parsed, target_set_size)


In [ ]:
words = GenerateAllEligibleWords()
wordVariables = GenerateVariablesForWords(words)
ruleClauses = GetRuleClauses(words, wordVariables)

solverPath = ""

In [ ]:
set_size = 10
sizeClauses, variableCount = GetSizeClauses(wordVariables, set_size)
clauses = ruleClauses + sizeClauses

Run(set_size, solverPath, variableCount, clauses)

In [ ]:
set_size = 5

sizeClauses, variableCount = GetSizeClauses(wordVariables, set_size)
clauses = ruleClauses + sizeClauses

Run(set_size, solverPath, variableCount, clauses)

In [ ]:
set_size = 20

sizeClauses, variableCount = GetSizeClauses(wordVariables, set_size)
clauses = ruleClauses + sizeClauses

Run(set_size, solverPath, variableCount, clauses)